# AES 一阶模板攻击：汉明重量（Hamming Weight）建模

本 notebook 是 `template_attack_demo.ipynb` 的**汉明重量建模**改写版：仍然是一阶模板攻击（多元高斯模板 + 最大似然判据），
但**模板不再按 256 个中间值分类，而是按中间值的汉明重量分成 9 类**。

## 核心改动

| | 值模板（原 notebook） | HW 模板（本 notebook） |
|---|---|---|
| 分类依据 | $v=\mathrm{Sbox}[p\oplus k]$，256 类 | $\mathrm{HW}(v)\in\{0,\dots,8\}$，9 类 |
| 模板数量 | 256 个 $\mathcal{N}(\mu_v,\Sigma_v)$ | 9 个 $\mathcal{N}(\mu_{hw},\Sigma_{hw})$ |
| 判决 | $\arg\max_k\sum_i\log\mathcal{N}(\mathbf t_i;\mu_{v_i},\Sigma_{v_i})$ | $\arg\max_k\sum_i\log\mathcal{N}(\mathbf t_i;\mu_{\mathrm{HW}(v_i)},\Sigma_{\mathrm{HW}(v_i)})$ |
| 建模迹需求 | 2400（≈9 条/类，已接近下限） | 800（≈90 条/类，很宽裕） |
| 隐含假设 | 无参数假设 | 泄漏近似只依赖 HW：$L=a\cdot\mathrm{HW}(v)+b+\text{noise}$ |

## 为什么这样还能攻击成功

HW 建模把"泄漏模型"与"噪声为多元高斯"两个假设合并：**同一 HW 的所有中间值共用一套高斯分布**。
它放弃了区分"同 HW 不同值"的能力，换来两个实际好处：

1. **每类样本数提升一个数量级** → 协方差矩阵 $\Sigma_{hw}$ 估计得准。小样本协方差估计不准，是模板攻击最容易翻车的地方（原 notebook 里 1500 条建模迹只能恢复 7/16 个字节，就是这个原因）。
2. 建模成本从 256 个协方差矩阵降到 9 个。

代价是**信息损失**：HW 相同的中间值被折叠到同一类，密钥猜测之间的似然差变小。
建模迹不足或噪声较大时，正确密钥可能排到第 2、第 3 名，或被"同 HW 邻域"的猜测反超。
末尾的对比实验会量化这一点。

## 流程

1. 建模集 / 攻击集划分（互不重叠）。
2. 用 **HW 标签**算 SNR，每字节挑 4 个 POI。
3. 每字节建 9 个多元高斯模板 $\mathcal{N}(\mu_{hw},\Sigma_{hw})$。
4. 攻击：对每个猜测 $k$ 计算 $\mathrm{HW}(\mathrm{Sbox}[p\oplus k])$，累加对数似然，取最大者。

In [ ]:
"""汉明重量模板攻击：常量与工具函数"""

import math

import numpy as np
import matplotlib.pyplot as plt
import estraces

plt.rcParams["figure.max_open_warning"] = 0
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "PingFang SC", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False      # 让负号正常显示

BYTES = 16                  # AES-128 的 16 个子密钥
N_HW = 9                    # HW(v) ∈ {0, 1, ..., 8}  ->  9 个模板
N_VAL = 256                 # 中间值本身（只在对比实验里用）

# ---------------------------- AES S-BOX ----------------------------
SBOX = np.array((
    0x63, 0x7c, 0x77, 0x7b, 0xf2, 0x6b, 0x6f, 0xc5, 0x30, 0x01, 0x67, 0x2b, 0xfe, 0xd7, 0xab, 0x76,
    0xca, 0x82, 0xc9, 0x7d, 0xfa, 0x59, 0x47, 0xf0, 0xad, 0xd4, 0xa2, 0xaf, 0x9c, 0xa4, 0x72, 0xc0,
    0xb7, 0xfd, 0x93, 0x26, 0x36, 0x3f, 0xf7, 0xcc, 0x34, 0xa5, 0xe5, 0xf1, 0x71, 0xd8, 0x31, 0x15,
    0x04, 0xc7, 0x23, 0xc3, 0x18, 0x96, 0x05, 0x9a, 0x07, 0x12, 0x80, 0xe2, 0xeb, 0x27, 0xb2, 0x75,
    0x09, 0x83, 0x2c, 0x1a, 0x1b, 0x6e, 0x5a, 0xa0, 0x52, 0x3b, 0xd6, 0xb3, 0x29, 0xe3, 0x2f, 0x84,
    0x53, 0xd1, 0x00, 0xed, 0x20, 0xfc, 0xb1, 0x5b, 0x6a, 0xcb, 0xbe, 0x39, 0x4a, 0x4c, 0x58, 0xcf,
    0xd0, 0xef, 0xaa, 0xfb, 0x43, 0x4d, 0x33, 0x85, 0x45, 0xf9, 0x02, 0x7f, 0x50, 0x3c, 0x9f, 0xa8,
    0x51, 0xa3, 0x40, 0x8f, 0x92, 0x9d, 0x38, 0xf5, 0xbc, 0xb6, 0xda, 0x21, 0x10, 0xff, 0xf3, 0xd2,
    0xcd, 0x0c, 0x13, 0xec, 0x5f, 0x97, 0x44, 0x17, 0xc4, 0xa7, 0x7e, 0x3d, 0x64, 0x5d, 0x19, 0x73,
    0x60, 0x81, 0x4f, 0xdc, 0x22, 0x2a, 0x90, 0x88, 0x46, 0xee, 0xb8, 0x14, 0xde, 0x5e, 0x0b, 0xdb,
    0xe0, 0x32, 0x3a, 0x0a, 0x49, 0x06, 0x24, 0x5c, 0xc2, 0xd3, 0xac, 0x62, 0x91, 0x95, 0xe4, 0x79,
    0xe7, 0xc8, 0x37, 0x6d, 0x8d, 0xd5, 0x4e, 0xa9, 0x6c, 0x56, 0xf4, 0xea, 0x65, 0x7a, 0xae, 0x08,
    0xba, 0x78, 0x25, 0x2e, 0x1c, 0xa6, 0xb4, 0xc6, 0xe8, 0xdd, 0x74, 0x1f, 0x4b, 0xbd, 0x8b, 0x8a,
    0x70, 0x3e, 0xb5, 0x66, 0x48, 0x03, 0xf6, 0x0e, 0x61, 0x35, 0x57, 0xb9, 0x86, 0xc1, 0x1d, 0x9e,
    0xe1, 0xf8, 0x98, 0x11, 0x69, 0xd9, 0x8e, 0x94, 0x9b, 0x1e, 0x87, 0xe9, 0xce, 0x55, 0x28, 0xdf,
    0x8c, 0xa1, 0x89, 0x0d, 0xbf, 0xe6, 0x42, 0x68, 0x41, 0x99, 0x2d, 0x0f, 0xb0, 0x54, 0xbb, 0x16), dtype=np.uint8)

# HW 查找表：HW_LUT[x] = popcount(x)，0..255 -> 0..8
HW_LUT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

# 两类"标签映射"：把 Sbox 输出映射成模板类别索引
CLASS_MAP_HW = HW_LUT                     # 256 个值 -> 9 个 HW 类
CLASS_MAP_VAL = np.arange(256)            # 恒等映射 -> 256 个值类（对比实验用）


def sub_bytes(plaintext, key_guess):
    """AES 字节替换：Sbox[plaintext ^ key_guess]，即建模用的中间值 v"""
    return SBOX[np.asarray(plaintext, dtype=np.uint8) ^ np.uint8(key_guess)]


def hamming_weight(values):
    """查表得到汉明重量；输入可以是数组，返回同样的形状"""
    return HW_LUT[np.asarray(values, dtype=np.uint8)]


def downsample_mean(samples, factor=10):
    """每 factor 个采样点取均值，用于降采样（降低内存与计算量）"""
    n, p = samples.shape
    return np.asarray(samples).reshape(n, p // factor, factor).mean(axis=2, dtype=np.float64)


def compute_snr(x, labels, n_classes):
    """逐采样点计算 SNR = Var_v(E[t|v]) / E_v(Var(t|v))

    x        : (n_traces, n_points) 建模迹
    labels   : (n_traces,) 每条迹的类别索引（HW 版是 0..8，值版是 0..255）
    n_classes: 类别个数
    返回     : (n_points,) SNR 曲线，值越大说明该点的类别可分性越强
    """
    x = np.asarray(x, dtype=np.float64)
    n = x.shape[0]
    onehot = np.zeros((n, n_classes), dtype=np.float64)
    onehot[np.arange(n), labels] = 1.0
    counts = np.maximum(onehot.sum(axis=0), 1.0)

    means = (onehot.T @ x) / counts[:, None]              # E[t|v]
    sq = (onehot.T @ (x * x)) / counts[:, None]
    var_in = np.maximum(sq - means ** 2, 0.0)             # Var(t|v)

    w = counts / counts.sum()
    mu = (means * w[:, None]).sum(axis=0)
    var_between = (((means - mu) ** 2) * w[:, None]).sum(axis=0)
    var_within = (var_in * w[:, None]).sum(axis=0)
    return var_between / (var_within + 1e-12)


def select_poi(snr_curve, n_poi=4, min_gap=5):
    """按 SNR 从高到低挑 POI，并保证相邻 POI 至少相隔 min_gap 个点"""
    pois = []
    for idx in np.argsort(snr_curve)[::-1]:
        if all(abs(int(idx) - q) > min_gap for q in pois):
            pois.append(int(idx))
        if len(pois) == n_poi:
            break
    return np.array(sorted(pois))


def build_templates(x, labels, n_classes, shrinkage=0.2, min_traces=None):
    """为每个类别估计一个多元高斯模板 N(mu_c, Sigma_c)

    x        : (n_traces, n_poi) 建模迹在 POI 上的取值
    labels   : (n_traces,) 类别索引（HW 版即 0..8）
    shrinkage: 协方差向「池化协方差」收缩的系数（小样本下显著提升稳定性）

    两种小样本处理：
      * 样本充足的类：Sigma_c 向池化协方差 Sigma_pool 收缩
      * 样本不足的类（n_c < min_traces）：直接用 Sigma_pool 作协方差，均值仍用该类的
      * **空类**（这一类一条建模迹都没有）：退化为「背景模板」——均值取全局均值、
        协方差取池化协方差。若像值模板那样记 -inf，则只要有一条攻击迹落进空类，
        这个密钥猜测就被直接判死，而 HW=0/8 这类稀有类别在最坏情况下确实可能是空的。

    返回 dict：
      means / precisions(=Sigma^-1) / consts / counts
      其中 consts[c] = -0.5*log|Sigma_c| - 0.5*mu_c^T Sigma_c^-1 mu_c
    """
    x = np.asarray(x, dtype=np.float64)
    d = x.shape[1]
    if min_traces is None:
        min_traces = d + 2                                # 至少要能估出 d 维协方差

    counts = np.bincount(labels, minlength=n_classes)
    means = np.zeros((n_classes, d))
    covs = [None] * n_classes
    for c in range(n_classes):
        idx = np.flatnonzero(labels == c)
        if idx.size == 0:
            continue
        xc = x[idx]
        means[c] = xc.mean(axis=0)
        if idx.size > 1:
            covs[c] = np.atleast_2d(np.cov(xc, rowvar=False))

    # 池化协方差：各类协方差的加权平均
    acc, wsum = np.zeros((d, d)), 0.0
    for c in range(n_classes):
        if covs[c] is not None:
            acc += covs[c] * counts[c]
            wsum += counts[c]
    pooled = acc / max(wsum, 1.0)
    pooled += np.eye(d) * (1e-9 * np.trace(pooled) / d + 1e-12)

    seen = counts > 0
    mu_global = (means[seen] * counts[seen, None]).sum(0) / counts[seen].sum()

    means_out = np.zeros((n_classes, d))
    precisions = np.zeros((n_classes, d, d))
    consts = np.full(n_classes, -np.inf)
    for c in range(n_classes):
        if counts[c] == 0:                                # 空类 -> 背景模板
            mu_c, cov = mu_global, pooled
        else:
            mu_c = means[c]
            own = covs[c] if (counts[c] >= min_traces and covs[c] is not None) else pooled
            cov = (1.0 - shrinkage) * own + shrinkage * pooled
        cov = cov + np.eye(d) * (1e-9 * np.trace(cov) / d + 1e-12)   # 保证正定可逆

        logdet = np.linalg.slogdet(cov)[1]
        prec = np.linalg.inv(cov)
        means_out[c] = mu_c
        precisions[c] = prec
        consts[c] = -0.5 * logdet - 0.5 * mu_c @ prec @ mu_c

    return dict(means=means_out, precisions=precisions, consts=consts, counts=counts)


def loglik_matrix(template, x, n_classes):
    """每条迹在各模板下的对数似然，返回 (n_traces, n_classes)

    log p(t|c) = -0.5*log|S_c| - 0.5*(t-mu_c)^T S_c^-1 (t-mu_c)
               = consts[c] + t^T S_c^-1 mu_c - 0.5 * t^T S_c^-1 t
    """
    ll = np.zeros((x.shape[0], n_classes))
    for c in range(n_classes):
        prec = template["precisions"][c]
        quad = np.einsum("ij,jk,ik->i", x, prec, x)
        ll[:, c] = template["consts"][c] + x @ (prec @ template["means"][c]) - 0.5 * quad
    return ll


def score_key_guesses(ll, plaintext_col, class_map):
    """把逐迹似然按密钥猜测聚合，返回 (最终得分 (256,), 累积得分 (256, n_traces))

    class_map: 把 Sbox 输出映射成模板类别索引
               —— CLASS_MAP_HW 时只用到 HW，CLASS_MAP_VAL 时是完整值
    对猜测 k：c_i = class_map[Sbox[p_i ^ k]]，得分 = sum_i log p(t_i | c_i)
    """
    class_map = np.asarray(class_map)
    n = ll.shape[0]
    cols = np.arange(n)[None, :]
    guesses = np.arange(256)[:, None]
    cls = class_map[sub_bytes(plaintext_col[None, :], guesses)].astype(np.intp)   # (256, n)
    cum = np.cumsum(ll[cols, cls], axis=1)
    return cum[:, -1], cum

## 1. 加载数据并划分建模集 / 攻击集

数据集 `ds_atmega2560_aes_power.ets`：**3000 条迹 × 50000 采样点**，明文随机，密钥 `00 01 02 ... 0f`。

HW 只有 9 个类别，建模迹需求比值模板低得多，这里取 **1600 条建模 + 1400 条攻击**：
1600/9 ≈ 178 条/类，足以把 4 维协方差估得比较稳。（值模板要 2400 条才够，也就是 9 条/类。）

In [ ]:
# ------------------------- 1. 加载数据并划分建模集 / 攻击集 -------------------------
DS_NAME = "ds_atmega2560_aes_power.ets"
NUM_POINTS = 50000         # 每条迹的原始采样点数
COMPRESS = 10              # 降采样倍数
N_PROFILING = 1600         # 建模迹数量（HW 9 类 -> 平均 178 条/类）
N_ATTACK = 1400            # 攻击迹数量
SEED = 0

print(f"加载数据集：{DS_NAME}")
ths = estraces.read_ths_from_ets_file(DS_NAME)
NUM_TRACES = len(ths)

# 降采样：每 COMPRESS 个点取平均值 -> 5000 个点
raw = ths.samples[:NUM_TRACES, :NUM_POINTS]
traces = downsample_mean(raw, COMPRESS)
del raw

plaintext = np.asarray(ths.metadatas["plaintext"][:NUM_TRACES], dtype=np.uint8)
correct_key = np.asarray(ths.metadatas["key"][0], dtype=np.uint8)

# 打乱后切分：模板攻击必须用「两批互不重叠」的迹
perm = np.random.RandomState(SEED).permutation(traces.shape[0])
traces, plaintext = traces[perm], plaintext[perm]

traces_prof, pt_prof = traces[:N_PROFILING], plaintext[:N_PROFILING]
traces_atk, pt_atk = traces[N_PROFILING:N_PROFILING + N_ATTACK], plaintext[N_PROFILING:N_PROFILING + N_ATTACK]

print("总迹数:", traces.shape[0], "| 采样点数:", traces.shape[1])
print("建模迹:", traces_prof.shape, " 攻击迹:", traces_atk.shape)
print("明文:", plaintext.shape, "| 正确密钥:", " ".join(f"{b:02x}" for b in correct_key))

# HW 的理论分布：P(HW = h) = C(8, h) / 2^8 —— 决定每类能分到多少建模迹
prob = np.array([math.comb(8, h) for h in range(N_HW)]) / 256
print("\nHW 分布：", "  ".join(f"HW={h}: {p*100:4.1f}%" for h, p in enumerate(prob)))

## 2. POI 选择：用 HW 标签算 SNR

和值模板一样，我们仍然用 SNR 找泄漏点，**区别只在于分组方式**：现在按 9 个 HW 类别算
"类间方差 / 类内方差"，而不是按 256 个中间值。

注意两种分组选出的 POI 位置几乎相同——泄漏点由硬件决定，与我们的标签无关；
但 HW 分组的 SNR 数值更低，因为 9 个类把 256 个值混在了一起，类间差异被"平均"掉了一部分。

In [ ]:
# ------------------------- 2. POI 选择（基于 HW 标签的 SNR） -------------------------
N_POI = 4          # 每个子密钥使用的 POI 个数（即模板维度）
MIN_GAP = 5        # 相邻 POI 的最小间隔（避免取到同一泄漏尖峰的邻近点）
SHRINKAGE = 0.2    # 协方差收缩系数（向池化协方差收缩）

# 以 byte 0 为例，对比「HW 标签(9 类)」与「值标签(256 类)」两种 SNR
v0 = sub_bytes(pt_prof[:, 0], correct_key[0])
snr_hw0 = compute_snr(traces_prof, hamming_weight(v0), N_HW)
snr_val0 = compute_snr(traces_prof, v0, N_VAL)
pois_hw0 = select_poi(snr_hw0, N_POI, MIN_GAP)
pois_val0 = select_poi(snr_val0, N_POI, MIN_GAP)

print("HW  标签选出的 POI：", pois_hw0, " SNR =", np.round(snr_hw0[pois_hw0], 3))
print("值  标签选出的 POI：", pois_val0, " SNR =", np.round(snr_val0[pois_val0], 3))

fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(12, 6), sharex=True)
axes[0].plot(traces_prof.mean(axis=0), color="0.6", lw=0.8, label="平均迹")
axes[0].set_title("平均功耗迹（降采样后）")
axes[0].set_ylabel("功耗")

axes[1].plot(snr_val0 / snr_val0.max(), color="0.7", lw=0.9, label="SNR（256 值标签，归一化）")
axes[1].plot(snr_hw0 / snr_hw0.max(), color="tab:blue", lw=1.2, label="SNR（9 个 HW 标签，归一化）")
axes[1].plot(pois_hw0, (snr_hw0 / snr_hw0.max())[pois_hw0], "rv", ms=10, label="选中的 POI")
axes[1].set_title("Byte 0 的 SNR 曲线：两种标签的峰值位置一致（泄漏点由硬件决定）")
axes[1].set_xlabel("采样点")
axes[1].set_ylabel("SNR（归一化）")
axes[1].legend(loc="upper right")
fig.tight_layout()
plt.show()

## 3. 建模：每个字节 9 个 HW 模板

对每个字节，把建模迹按 $\mathrm{HW}(\mathrm{Sbox}[p_i\oplus k])$ 分成 9 组，
在选定的 4 个 POI 上估计 $\mu_{hw}\in\mathbb{R}^4$ 与 $\Sigma_{hw}\in\mathbb{R}^{4\times4}$。

In [ ]:
# ------------------------- 3. 建模：每字节 9 个 HW 模板 -------------------------
snr_curves = np.zeros((BYTES, traces_prof.shape[1]))
pois = np.zeros((BYTES, N_POI), dtype=int)
labels_prof = np.zeros((BYTES, N_PROFILING), dtype=np.uint8)   # 建模迹的 HW 标签
templates = []

for wi in range(BYTES):
    labels_prof[wi] = hamming_weight(sub_bytes(pt_prof[:, wi], correct_key[wi]))
    snr_curves[wi] = compute_snr(traces_prof, labels_prof[wi], N_HW)
    pois[wi] = select_poi(snr_curves[wi], N_POI, MIN_GAP)
    x = traces_prof[:, pois[wi]]                      # (n_prof, N_POI)
    tpl = build_templates(x, labels_prof[wi], N_HW, shrinkage=SHRINKAGE)
    templates.append(tpl)
    c = tpl["counts"]
    print(f"byte {wi:2d}: POI {pois[wi]}  各类样本数 {np.min(c)}~{np.max(c)}"
          f"  空类 {int((c == 0).sum())}  总样本 {c.sum()}")

print("\nbyte 0 的 9 个 HW 类别样本数：", templates[0]["counts"])

### 模板长什么样：均值向量随 HW 单调变化

泄漏确实与 HW 近似线性吗？看两件事就够：

1. 9 个 HW 类别在泄漏窗口内的平均迹应该**按 HW 值整齐排列**（HW 越大，翻转位数越多，功耗越高/越低，取决于实现）；
2. 在每个 POI 上，$\mu_{hw}$ 关于 $hw$ 的散点应该近似落在一条直线上（用相关系数 $r$ 定量）。

本数据集的泄漏方向为**负**（HW 越大，均值越低）——符号无所谓，重要的是 9 条曲线分层有序、$|r|$ 接近 1，
这说明 HW 线性泄漏假设在本数据集上成立。

In [ ]:
# ------------------------- 模板可视化：byte 0 -------------------------
lo = max(0, pois[0][0] - 15)
hi = min(traces_prof.shape[1], pois[0][0] + 16)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
cmap = plt.get_cmap("viridis")
for hw in range(N_HW):
    idx = np.flatnonzero(labels_prof[0] == hw)
    if idx.size == 0:
        continue
    axes[0].plot(np.arange(lo, hi), traces_prof[idx, lo:hi].mean(axis=0),
                 color=cmap(hw / (N_HW - 1)), lw=1.4, label=f"HW={hw} (n={idx.size})")
for p in pois[0]:
    if lo <= p < hi:
        axes[0].axvline(p, color="red", ls=":", lw=1.2)
axes[0].set_title("Byte 0：9 个 HW 模板的平均迹（红虚线 = POI，曲线按 HW 分层）")
axes[0].set_xlabel("采样点")
axes[0].set_ylabel("功耗")
axes[0].legend(fontsize=7, ncol=2)

mu0 = templates[0]["means"]                          # (9, N_POI)
hw_axis = np.arange(N_HW)
for j in range(min(2, N_POI)):                       # 取前两个 POI 举例
    y = mu0[:, j]
    k, b = np.polyfit(hw_axis, y, 1)
    r = np.corrcoef(hw_axis, y)[0, 1]
    axes[1].plot(hw_axis, y, "o-", lw=0.8, ms=5,
                 label=f"POI {pois[0][j]}: 拟合斜率 {k:.3g}, r = {r:.3f}")
    axes[1].plot(hw_axis, k * hw_axis + b, ls="--", lw=0.8, color="0.5")
axes[1].set_title("模板均值 μ_hw 关于 HW 的线性度")
axes[1].set_xlabel("HW")
axes[1].set_ylabel("μ_hw")
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

# 各类样本数分布
plt.figure(figsize=(9, 3))
plt.bar(np.arange(N_HW), templates[0]["counts"], width=0.85, color="tab:blue")
plt.axhline(N_PROFILING / N_HW, color="tab:red", ls="--", lw=1, label=f"平均值 {N_PROFILING/N_HW:.0f}")
plt.title("Byte 0：每个 HW 类别分到的建模迹条数")
plt.xlabel("HW")
plt.ylabel("建模迹条数")
plt.xticks(np.arange(N_HW))
plt.legend()
plt.tight_layout()
plt.show()

## 4. 攻击：最大似然恢复密钥

对每个猜测 $k$，把每条攻击迹的似然按 $\mathrm{HW}(\mathrm{Sbox}[p_i\oplus k])$ 查表累加：

$$\hat k=\arg\max_k \sum_i \log p\left(\mathbf t_i \mid c=\mathrm{HW}(\mathrm{Sbox}[p_i\oplus k])\right)$$

与值模板唯一的代码差别就是 `class_map` 从恒等映射换成 `HW_LUT`。

In [ ]:
# ------------------------- 4. 攻击：最大似然估计密钥 -------------------------
recovered = np.zeros(BYTES, dtype=int)
ranks = np.zeros(BYTES, dtype=int)          # 正确密钥在得分排序中的名次（0 = 第一）
min_traces = np.zeros(BYTES, dtype=int)     # 累积多少条迹后首次命中正确密钥
best_over_time = np.zeros((BYTES, N_ATTACK), dtype=int)
scores_all = np.zeros((BYTES, 256))

for wi in range(BYTES):
    x = traces_atk[:, pois[wi]].astype(np.float64)
    ll = loglik_matrix(templates[wi], x, N_HW)              # (n_atk, 9)
    scores, cum = score_key_guesses(ll, pt_atk[:, wi], CLASS_MAP_HW)
    scores_all[wi] = scores

    recovered[wi] = int(np.argmax(scores))
    ranks[wi] = int((scores > scores[correct_key[wi]]).sum())

    best = np.argmax(cum, axis=0)
    best_over_time[wi] = best
    hit = np.flatnonzero(best == correct_key[wi])
    min_traces[wi] = int(hit[0] + 1) if hit.size else -1
    print(f"byte {wi:2d}: recovered = 0x{recovered[wi]:02x}, rank = {ranks[wi]:3d}, "
          f"命中所需迹数 = {min_traces[wi]}")

# 每个字节的得分曲线（红色竖线 = 正确密钥）
fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(14, 9))
for wi in range(BYTES):
    ax = axes[wi // 4, wi % 4]
    s = scores_all[wi] - np.max(scores_all[wi])
    ax.plot(s, lw=0.8, color="tab:blue")
    ax.axvline(int(correct_key[wi]), color="red", ls="--", lw=1.2)
    ax.set_title(f"byte {wi}: rank {ranks[wi]}", fontsize=9)
    ax.set_xlim(0, 255)
    ax.tick_params(labelsize=7)
fig.suptitle("各字节的对数似然得分（相对最大值），红色虚线 = 正确密钥", fontsize=12)
fig.tight_layout()
plt.show()

## 5. 结果汇总

HW 模板的似然区分度比值模板弱：相邻 HW 类别的模板均值只差"一位翻转"的功耗，
噪声稍大就会混。所以除了最终得分，还关注两件事：
- **rank**：正确密钥排第几（rank=0 即第一）；
- **命中所需迹数**：累积多少条攻击迹后第一次排到第一，反映"证据积累速度"。

典型现象：多数字节几十条迹内就稳定命中，但个别字节（本次运行是 byte 12）要几百条才收敛——
这正是 HW 折叠造成的区分度损失，也是下一节对比实验里"值模板上限更高"的原因。

In [ ]:
# ------------------------- 5. 结果汇总与可视化 -------------------------
def display_key_analysis(recovered_keys, correct_keys):
    """对比并展示恢复的密钥与正确密钥的差异。"""
    print("\n" + "=" * 65)
    print(f"{'密钥字节分析 (Key Byte Analysis)':^60}")
    print("=" * 65)
    print(f"{'Idx':<6} | {'Recovered':<12} | {'Status':<8} | {'Correct':<12}")
    print("-" * 65)
    correct_count = 0
    for i in range(len(correct_keys)):
        rec_byte = recovered_keys[i] if i < len(recovered_keys) else None
        cor_byte = correct_keys[i]
        rec_str = f"0x{rec_byte:02x}" if rec_byte is not None else "None"
        cor_str = f"0x{cor_byte:02x}"
        if rec_byte == cor_byte:
            status = "✅ Match"
            correct_count += 1
        else:
            status = "❌ Error"
        print(f"{i:<6} | {rec_str:<12} | {status:<8} | {cor_str:<12}")
    print("-" * 65)
    total = len(correct_keys)
    accuracy = (correct_count / total) * 100 if total > 0 else 0
    print(f"\n📊 统计结果:")
    print(f"   总字节数: {total}")
    print(f"   正确匹配: {correct_count}")
    print(f"   攻击准确率: {accuracy:.1f}%")
    if accuracy == 100.0:
        print("   🎉 恭喜！所有子密钥均已成功恢复！")
    else:
        print("   ⚠️ 仍有部分密钥未恢复或错误。")
    print("=" * 65 + "\n")


print("The recovered Key is: ")
for b in recovered:
    print("%02x " % b, end="")
print("\nThe correct Key is: ")
for b in correct_key:
    print("%02x " % b, end="")
print()

display_key_analysis(recovered, correct_key)

# 需要多少条攻击迹才能恢复全部子密钥？
n_recovered = (best_over_time == correct_key[:, None]).sum(axis=0)

plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, N_ATTACK + 1), n_recovered, color="tab:red", lw=2)
plt.axhline(BYTES, color="0.5", ls="--", lw=1, label="16 个子密钥全部恢复")
plt.xlabel("使用的攻击迹条数")
plt.ylabel("已恢复的子密钥个数")
plt.title("HW 模板攻击的密钥恢复曲线（success rate vs. #traces）")
plt.ylim(-0.5, BYTES + 0.5)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print("每个字节所需迹数（首次命中正确密钥）：", min_traces)
print("每个字节的密钥排名（0 = 排名第一）：", ranks)

## 6. 对比实验：HW 模型省多少建模迹？

同一份数据、同一套 POI 选择逻辑、同一个判据，只改 `class_map`：

- `CLASS_MAP_HW`：9 类 HW 模板；
- `CLASS_MAP_VAL`：256 类值模板（原 notebook 的做法）。

固定攻击迹为 1400 条，逐步增加建模迹，看能恢复多少个子密钥。这一格要跑十几秒。

In [ ]:
# ------------------------- 6. 对比实验：HW(9 类) vs 值(256 类) -------------------------
def run_attack(tr_prof, pt_prof_, tr_atk, pt_atk_, class_map, n_classes, key,
               n_poi=N_POI, min_gap=MIN_GAP, shrinkage=SHRINKAGE):
    """一把完整的模板攻击，返回恢复结果（供对比实验调用）"""
    rec = np.zeros(BYTES, dtype=int)
    rk = np.zeros(BYTES, dtype=int)
    for wi in range(BYTES):
        lab = np.asarray(class_map)[sub_bytes(pt_prof_[:, wi], key[wi])]
        p = select_poi(compute_snr(tr_prof, lab, n_classes), n_poi, min_gap)
        tpl = build_templates(tr_prof[:, p], lab, n_classes, shrinkage=shrinkage)
        ll = loglik_matrix(tpl, tr_atk[:, p].astype(np.float64), n_classes)
        sc, _ = score_key_guesses(ll, pt_atk_[:, wi], class_map)
        rec[wi] = int(np.argmax(sc))
        rk[wi] = int((sc > sc[key[wi]]).sum())
    return rec, rk


N_PROF_LIST = [200, 400, 600, 800, 1200, 1600]
tr_atk_cmp, pt_atk_cmp = traces[N_PROFILING:], plaintext[N_PROFILING:]   # 所有配置共用同一批攻击迹

res = {"HW": [], "值": []}
print(f"{'建模迹':>6} | {'值(256 类)':>18} | {'HW(9 类)':>18}")
print("-" * 52)
for np_ in N_PROF_LIST:
    tr_p, pt_p = traces[:np_], plaintext[:np_]
    rec_hw, _ = run_attack(tr_p, pt_p, tr_atk_cmp, pt_atk_cmp, CLASS_MAP_HW, N_HW, correct_key)
    rec_val, _ = run_attack(tr_p, pt_p, tr_atk_cmp, pt_atk_cmp, CLASS_MAP_VAL, N_VAL, correct_key)
    res["HW"].append(int((rec_hw == correct_key).sum()))
    res["值"].append(int((rec_val == correct_key).sum()))
    print(f"{np_:>6} | {res['值'][-1]:>8}/16 ({np_/256:4.1f} 条/类) | "
          f"{res['HW'][-1]:>8}/16 ({np_/N_HW:5.1f} 条/类)")

plt.figure(figsize=(8, 4.5))
plt.plot(N_PROF_LIST, res["值"], "o-", color="tab:gray", lw=2, label="值模板（256 类）")
plt.plot(N_PROF_LIST, res["HW"], "s-", color="tab:blue", lw=2, label="HW 模板（9 类）")
plt.axhline(BYTES, color="0.5", ls="--", lw=1, label="16 个子密钥全部恢复")
plt.xlabel("建模迹条数（profiling traces）")
plt.ylabel("成功恢复的子密钥个数")
plt.title("同样的建模迹预算下，HW 建模的恢复率更高")
plt.xticks(N_PROF_LIST)
plt.ylim(-0.5, BYTES + 0.5)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 7. 小结：HW 建模的取舍

**收益**

- 每类样本数从个位数涨到上百条，$\Sigma$ 估得准。这是模板攻击能否成功的第一要素。
- 建模集、存储、求逆的开销都从 256 份降到 9 份，选 POI 前可以先粗筛。
- 对"泄漏近似线性"的实现（大多数软件 AES、8 位 MCU 的寄存器/总线泄漏）非常划算：
  实测 800 条建模迹即可恢复 16/16，而值模板在同样预算下只能恢复一半左右，要 1600~2400 条才行。

**代价**

- **信息损失**：$\mathrm{HW}$ 相同的中间值被折叠，似然差变小。建模迹偏少时，正确密钥可能被
  "同 HW 邻域"的猜测反超（rank 掉到 1~5），需要更多**攻击迹**才收敛。
- **模型失配**：如果泄漏不是 HW 的线性函数（例如带强烈数据依赖的开关噪声、位反转编码、
  掩码防护），HW 模板会系统性偏移，此时要么回到 256 值模板，要么做二阶/多变量分析。
- HW 类别的样本数天然不均（$\mathrm{HW}=0/8$ 各占 0.39%），1600 条建模迹里 HW=0 可能只有 3~7 条，
  协方差退化为池化协方差——本 notebook 已用「背景模板 + 收缩」兜住这种情况。

**继续往下走的方向**

| 想改进什么 | 做法 |
|---|---|
| 少用建模迹、还要精度 | 用 HW 模板做**粗筛**，再用少量值模板在候选集内**精筛**（两级模板） |
| 更贴合线性泄漏 | 用线性回归 / LDA 代替逐类高斯：$\hat L = a\cdot\mathrm{HW}(v)+b$，等价于 CPA，但可用模板框架解释 |
| 每字节只用 1~2 个点 | 用单比特模板（9 类 → 2 类），样本更多但信息更少 |
| 抗掩码 | 二阶模板攻击：对两个时间点的联合分布建模，类别用 $\mathrm{HW}(v)$ 仍可沿用 |
| 减少攻击迹 | 多字节联合似然（同时枚举两个子密钥），代价是计算量 $256^2$ |

**调参提示**

- `N_POI = 4`：HW 模型的似然区分度弱，维度再高收益不大（实测 3/4/6/8 差别在 1 个字节以内），
  但维度过高会让协方差估计变差。
- `SHRINKAGE = 0.2`：HW 类别样本多，收缩可以小一些；若建模迹 < 400 条，调到 0.4~0.5。
- 若某些字节 `rank != 0`：优先加**攻击迹**（似然需要积累），其次加建模迹；只有建模迹时收益递减很快。
- 模板可复用：换密钥后 HW 模板依然有效（原始建模集的 $p\oplus k$ 与攻击集的 $p'\oplus k'$ 只要落到同一 HW 类即可），
  这正是它比 CPA 更危险的地方。